# Result Tables (Table 1 & Table 2)

Generates LaTeX tables for the paper from `WikiEvalResult` pkl files.

- **Table 1**: Completeness (C) and G-BERTScore-R (G-R) for all models
- **Table 2**: Zero-shot vs ICL comparison (EDC+ only)

## Setup

Run from a salloc on rome (no GPU needed, ~16GB RAM):
```
salloc -p rome -n 1 --ntasks-per-node 1 --cpus-per-task 1 -t 1:59:00 --mem=16G
conda activate emerge
cd ~/repositories/emerge-stage
jupyter lab --no-browser --port=8888
```

In [1]:
import os
import sys
import pandas as pd

# Add src to path so we can import our modules
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath('.')), 'src'))
# Also handle case when running from the stats directory itself
sys.path.insert(0, os.path.join(os.path.abspath('../../..'), 'src'))

from stats.evaluation.load_results import (
    load_results,
    metrics_to_report_cie,
    models_to_load,
    model_name_to_latex,
    make_agg_and_agg_open,
    make_agg_all,
    make_metrics_latex_table,
)

## Configuration

**Three scoring modes are available:**

1. **Legacy scoring (reproduces ICML submission exactly):** uses the old pipeline cached
   pkls or the `20260324_*_legacy_with_kg` new-pipeline pkl. Both use `score_empty_predictions_as_zero: false`
   (the original, pre-rebuttal behavior where instances with no predictions were excluded
   from metric averages, inflating scores).

2. **Fixed scoring (post-rebuttal):** uses `20260324_*_fixed_with_kg` new-pipeline pkl.
   `score_empty_predictions_as_zero: true` — instances with no predictions are scored as
   P=0/R=0/F1=0 and included in averages. Scores are **lower** than legacy — this is correct.

3. **Old pipeline cached pkls:** set `USE_OLD_PIPELINE = True`. Loads pre-computed DataFrames
   from the old repo's cache. Always uses legacy scoring. Useful as a fallback if new-pipeline
   pkls are unavailable.

See `src/evaluation/README.md` for details on the scoring bug and config options.

In [2]:
# =====================================================================
# OPTION A: Old pipeline cached pkls (legacy scoring — reproduces paper exactly)
# =====================================================================
# To use: set USE_OLD_PIPELINE = True
OLD_PIPELINE_CACHE_DIR = '/path/to/data/wikipedia-processing/src/stats/cache_stats/'
OLD_PIPELINE_CONFIG_PATH = '../../experiments/s14_experiments_stats_v13_ipynb/20260122_all_stats_for_tiny_slurm/config.json'
# USE_OLD_PIPELINE = True

# =====================================================================
# OPTION B: New pipeline pkl (13 models including ZS)
# =====================================================================
# To use: set USE_OLD_PIPELINE = False and uncomment one PKL_PATH below.
USE_OLD_PIPELINE = False

# --- Legacy scoring + KG snapshots (13 models, reproduces paper values):
PKL_PATH = '/path/to/storage/emerge/output/experiments/s0x_evaluate_predictions/20260324_all_models_with_zs_legacy_with_kg/wiki_eval_result.pkl'
# --- Fixed scoring + KG snapshots (13 models, score_empty_predictions_as_zero=true):
# PKL_PATH = '/path/to/storage/emerge/output/experiments/s0x_evaluate_predictions/20260324_all_models_with_zs_fixed_with_kg/wiki_eval_result.pkl'
# --- Legacy scoring + KG snapshots (11 models only, no ZS — Table 2 will be empty):
# PKL_PATH = '/path/to/storage/emerge/output/experiments/s0x_evaluate_predictions/20260324_submitted_icml_legacy_with_kg/wiki_eval_result.pkl'

# Models to load — includes ZS models for Table 2
models_to_load_with_zs = {
    'rebel', 'relik-oie', 'relik-cie',
    "edc-plus-azure_ai/Mistral-small",
    "edc-plus-azure_ai/Mistral-Large-2411",
    "edc-plus-open-ai/gpt-5.1/non-canonicalized",
    'kg-gen/azure_ai/Mistral-small',
    'kg-gen/azure_ai/Mistral-Large-2411',
    'kg-gen/azure/gpt5.1',
    'rakg/azure_ai/Mistral-small',
    'rakg/azure_ai/Mistral-Large-2411',
    # ZS models (needed for Table 2):
    'edc-plus-zshot-open_ai/GPT-5_1',
    'edc-plus-zshot-azure_ai/Mistral-Large-2411',
}

# Assessor config — naming differs between pipelines
assessor_by_prompt_type_old = {
    'triple_assertion': 'Meta-Llama-3.1-405B_prompt_v1_triple_assertion',
    'triple_deprecation': 'Meta-Llama-3.1-405B_prompt_v1_triple_deprecation'
}
assessor_by_prompt_type_new = {
    'triple_assertion': 'Meta-Llama-3.1-405B_prompt_v1',
    'triple_deprecation': 'Meta-Llama-3.1-405B_prompt_v1'
}

In [3]:
import hashlib
import json as json_mod

if USE_OLD_PIPELINE:
    # Compute cache key (same logic as s14_load_results_v13.py)
    cache_key = hashlib.sha256(OLD_PIPELINE_CONFIG_PATH.encode("utf-8")).hexdigest()[:16]
    canonical = json_mod.dumps(assessor_by_prompt_type_old, sort_keys=True, separators=(",", ":"))
    assessor_tag = "assessor_" + hashlib.sha256(canonical.encode("utf-8")).hexdigest()[:12]
    prefix = cache_key + "_" + assessor_tag

    df_wiki_metrics_cie = pd.read_pickle(os.path.join(OLD_PIPELINE_CACHE_DIR, prefix + "_df_wiki_metrics_cie.pkl"))
    df_metrics_open_ie = pd.read_pickle(os.path.join(OLD_PIPELINE_CACHE_DIR, prefix + "_df_metrics_open_ie.pkl"))

    # Filter to models_to_load_with_zs
    df_wiki_metrics_cie = df_wiki_metrics_cie[df_wiki_metrics_cie['model'].isin(models_to_load_with_zs)]
    df_metrics_open_ie = df_metrics_open_ie[df_metrics_open_ie['model'].isin(models_to_load_with_zs)]

    print(f'Loaded from old pipeline cache: {OLD_PIPELINE_CACHE_DIR}')
    print(f'Cache prefix: {prefix}')
else:
    (df_wiki_metrics_cie, df_metrics_open_ie, df_preds_gt_cie,
     df_preds_open_ie, df_instances, df_additional_stats) = load_results(
        pkl_path=PKL_PATH,
        assessor_by_prompt_type=assessor_by_prompt_type_new,
        filter_models=models_to_load_with_zs,
    )

print(f'df_wiki_metrics_cie.shape: {df_wiki_metrics_cie.shape}')
print(f'df_metrics_open_ie.shape: {df_metrics_open_ie.shape}')
print(f'Models: {sorted(df_wiki_metrics_cie.model.unique().tolist())}')
print(f'Metrics: {sorted(df_wiki_metrics_cie.metric.unique().tolist())}')

df_wiki_metrics_cie.shape: (4835027, 10)
df_wiki_metrics_cie_filtered.shape: (4679031, 10)
df_preds_gt_cie.shape: (56686, 24)
df_preds_gt_cie_filtered.shape: (43563, 25)


df_wiki_metrics_cie.shape: (4948850, 12)
df_metrics_open_ie.shape: (919271, 12)


Models: ['edc-plus-azure_ai/Mistral-Large-2411', 'edc-plus-azure_ai/Mistral-small', 'edc-plus-open-ai/gpt-5.1/non-canonicalized', 'edc-plus-zshot-azure_ai/Mistral-Large-2411', 'edc-plus-zshot-open_ai/GPT-5_1', 'kg-gen/azure/gpt5.1', 'kg-gen/azure_ai/Mistral-Large-2411', 'kg-gen/azure_ai/Mistral-small', 'rakg/azure_ai/Mistral-Large-2411', 'rakg/azure_ai/Mistral-small', 'rebel', 'relik-cie', 'relik-oie']


Metrics: ['bleu-f1', 'bleu-precision', 'bleu-recall', 'bleu-triple', 'completeness', 'completeness_score', 'ent-coverage-all-f1', 'ent-coverage-all-precision', 'ent-coverage-all-recall', 'ent-coverage-emerg-f1', 'ent-coverage-emerg-precision', 'ent-coverage-emerg-recall', 'ent-coverage-exist-f1', 'ent-coverage-exist-precision', 'ent-coverage-exist-recall', 'entity_coverage', 'gj-f1', 'gj-precision', 'gj-recall', 'gj-triple', 'rouge-f1', 'rouge-precision', 'rouge-recall', 'rouge-triple']


## Aggregate metrics

In [4]:
spec = pd.DataFrame(metrics_to_report_cie)

agg, agg_open = make_agg_and_agg_open(
    df_wiki_metrics_cie=df_wiki_metrics_cie,
    df_metrics_open_ie=df_metrics_open_ie,
    spec=spec,
    groupby_cols=['tkgu_type', 'model', 'metric', 'evaluator_model']
)

agg_all = make_agg_all(agg, agg_open)
print(f'agg_all.columns: {agg_all.columns.tolist()}')
print(f'agg_all.metric.unique(): {agg_all.metric.unique()}')

No overlapping (metric, evaluator_model, model) between agg and agg_open.
sanity check, all have to be in 1: 1    220
Name: count, dtype: int64
agg_all.columns: ['tkgu_type', 'model', 'metric', 'evaluator_model', 'score', 'alias', 'group', 'multiply_by_100', 'score_scaled']
agg_all.metric.unique(): ['completeness' 'gj-f1' 'gj-precision' 'gj-recall']


## Table 1 — All models (C + G-R)

Completeness and G-BERTScore Recall for all evaluated IE models.

In [5]:
allowed_models_table1 = [
    # EDC+
    "edc-plus-open-ai/gpt-5.1/non-canonicalized",
    "edc-plus-azure_ai/Mistral-Large-2411",
    "edc-plus-azure_ai/Mistral-small",
    # KG-GEN
    'kg-gen/azure/gpt5.1',
    'kg-gen/azure_ai/Mistral-Large-2411',
    'kg-gen/azure_ai/Mistral-small',
    # RAKG
    'rakg/azure_ai/Mistral-Large-2411',
    'rakg/azure_ai/Mistral-small',
    # Baselines
    'relik-oie',
    'relik-cie',
    'rebel',
]

latex = make_metrics_latex_table(
    agg_all=agg_all,
    spec=spec,
    allowed_models=allowed_models_table1,
    model_name_to_latex=model_name_to_latex,
    show_aliases=["C", "G-R"],
)
print(latex)

\begin{table*}
\begin{center}
\begin{small}
  \begin{sc}
    \setlength{\tabcolsep}{3pt}
    \rowcolors{4}{rowwhite}{rowgray}
    \begin{tabular}{l|cc|cc|cc|cc|cc}
      \toprule
      & \multicolumn{2}{c|}{\opexists} & \multicolumn{2}{c|}{\opadd} & \multicolumn{2}{c|}{\opmintadd} & \multicolumn{2}{c|}{\opinfer} & \multicolumn{2}{c}{\opdeprecate} \\
      \cmidrule(lr){2-3}\cmidrule(lr){4-5}\cmidrule(lr){6-7}\cmidrule(lr){8-9}\cmidrule(lr){10-11}
      Model & C$\uparrow$ & G-R$\uparrow$ & C$\uparrow$ & G-R$\uparrow$ & C$\uparrow$ & G-R$\uparrow$ & C$\uparrow$ & G-R$\uparrow$ & C$\uparrow$ & G-R$\uparrow$ \\
\midrule
\cellcolor{archEDCp} EDC+ GPT-5.1 & \textbf{25.8} & \textbf{66.8} & \textbf{40.3} & \textbf{77.6} & \textbf{35.7} & \textbf{77.5} & \textbf{22.0} & \textbf{72.7} & \textbf{50.9} & \textbf{78.6} \\
\cellcolor{archEDCp} EDC+ M-Lg & 16.5 & 55.1 & \underline{30.7} & \underline{72.0} & \underline{28.4} & \underline{72.4} & 20.0 & 69.1 & \underline{45.3} & \underline{75.5} \\
\c

## Table 1 — Full version (C + P + R + F1)

In [6]:
latex_full = make_metrics_latex_table(
    agg_all=agg_all,
    spec=spec,
    allowed_models=allowed_models_table1,
    model_name_to_latex=model_name_to_latex,
    show_aliases=["C", "R", "P", "F1"],
)
print(latex_full)

\begin{table*}
\begin{center}
\begin{small}
  \begin{sc}
    \setlength{\tabcolsep}{3pt}
    \rowcolors{4}{rowwhite}{rowgray}
    \begin{tabular}{l|cccc|cccc|cccc|cccc|cccc}
      \toprule
      & \multicolumn{4}{c|}{\opexists} & \multicolumn{4}{c|}{\opadd} & \multicolumn{4}{c|}{\opmintadd} & \multicolumn{4}{c|}{\opinfer} & \multicolumn{4}{c}{\opdeprecate} \\
      \cmidrule(lr){2-5}\cmidrule(lr){6-9}\cmidrule(lr){10-13}\cmidrule(lr){14-17}\cmidrule(lr){18-21}
      Model & C$\uparrow$ & G-P$\uparrow$ & G-R$\uparrow$ & G-F1$\uparrow$ & C$\uparrow$ & G-P$\uparrow$ & G-R$\uparrow$ & G-F1$\uparrow$ & C$\uparrow$ & G-P$\uparrow$ & G-R$\uparrow$ & G-F1$\uparrow$ & C$\uparrow$ & G-P$\uparrow$ & G-R$\uparrow$ & G-F1$\uparrow$ & C$\uparrow$ & G-P$\uparrow$ & G-R$\uparrow$ & G-F1$\uparrow$ \\
\midrule
\cellcolor{archEDCp} EDC+ GPT-5.1 & \textbf{25.8} & 33.6 & \textbf{66.8} & 38.1 & \textbf{40.3} & 18.4 & \textbf{77.6} & 27.5 & \textbf{35.7} & 17.9 & \textbf{77.5} & 26.5 & \textbf{22.0} & 23.4 &

## Table 2 — Zero-shot vs ICL (EDC+ only)

In [7]:
allowed_models_table2 = [
    # EDC+ ICL
    "edc-plus-open-ai/gpt-5.1/non-canonicalized",
    "edc-plus-azure_ai/Mistral-Large-2411",
    # EDC+ Zero-shot
    "edc-plus-zshot-open_ai/GPT-5_1",
    'edc-plus-zshot-azure_ai/Mistral-Large-2411',
]

latex_zs = make_metrics_latex_table(
    agg_all=agg_all,
    spec=spec,
    allowed_models=allowed_models_table2,
    model_name_to_latex=model_name_to_latex,
    show_aliases=["C", "G-R"],
    highlight_best=False,
)
print(latex_zs)

\begin{table*}
\begin{center}
\begin{small}
  \begin{sc}
    \setlength{\tabcolsep}{3pt}
    \rowcolors{4}{rowwhite}{rowgray}
    \begin{tabular}{l|cc|cc|cc|cc|cc}
      \toprule
      & \multicolumn{2}{c|}{\opexists} & \multicolumn{2}{c|}{\opadd} & \multicolumn{2}{c|}{\opmintadd} & \multicolumn{2}{c|}{\opinfer} & \multicolumn{2}{c}{\opdeprecate} \\
      \cmidrule(lr){2-3}\cmidrule(lr){4-5}\cmidrule(lr){6-7}\cmidrule(lr){8-9}\cmidrule(lr){10-11}
      Model & C$\uparrow$ & G-R$\uparrow$ & C$\uparrow$ & G-R$\uparrow$ & C$\uparrow$ & G-R$\uparrow$ & C$\uparrow$ & G-R$\uparrow$ & C$\uparrow$ & G-R$\uparrow$ \\
\midrule
\cellcolor{archEDCp} EDC+ GPT-5.1 & 25.8 & 66.8 & 40.3 & 77.6 & 35.7 & 77.5 & 22.0 & 72.7 & 50.9 & 78.6 \\
\cellcolor{archEDCp} EDC+ M-Lg & 16.5 & 55.1 & 30.7 & 72.0 & 28.4 & 72.4 & 20.0 & 69.1 & 45.3 & 75.5 \\
\cellcolor{archEDCp} EDC+ ZS GPT 5.1 & 15.5 & 64.5 & 22.6 & 71.7 & 18.9 & 70.0 & 5.8 & 52.8 & 32.0 & 67.5 \\
\cellcolor{archEDCp} EDC+ ZS M-Lg & 9.8 & 57.0 & 15.6 &